# IV. backprop, by hand

Backpropagating an MLP — matmul, `tanh`, batch normalization, and cross-entropy — one gradient at a time, checking every one against PyTorch's `.grad`, then collapsing the cross-entropy and batch-norm blocks into single lines.

The write-up, the method, and the discussion of the results live on the project page; this notebook is the demonstration.

## Setup

Scaffolding inherited from part 3: read the names, build the character vocabulary, cut the dataset into train/dev/test.

In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
names = open('names.txt', 'r').read().splitlines()
print(f"total names in dataset: {len(names)}")

total names in dataset: 32033


In [3]:
# build character vocabulary and mapping
chars = sorted(list(set(''.join(names))))
ctoi = {c:i+1 for i,c in enumerate(chars)}
ctoi['.'] = 0
itoc = {i:c for c, i in ctoi.items()}
vocab_size = len(itoc)
print(f"characters: {vocab_size}")

characters: 27


In [4]:
# build the dataset
block_size = 3 # context length

def build_dataset(names):
    X, Y = [], []

    for n in names:
        context = [0] * block_size
        for ch in n + '.':
            ix = ctoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]

    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, Y.shape)
    return X, Y

import random
random.seed(1994)
random.shuffle(names)
n1 = int(0.8*len(names))
n2 = int(0.9*len(names))

print("Xtrn, Ytrn = 80%")
Xtrn, Ytrn = build_dataset(names[:n1])
print("Xdev, Ydev = 10%")
Xdev, Ydev = build_dataset(names[n1:n2])
print("Xtst, Ytst = 10%")
Xtst, Ytst = build_dataset(names[n2:])

Xtrn, Ytrn = 80%


torch.Size([182599, 3])

 torch.Size([182599])
Xdev, Ydev = 10%
torch.Size([22749, 3]) torch.Size([22749])
Xtst, Ytst = 10%
torch.Size([22798, 3]) torch.Size([22798])


### The gradient checker

`cmp` compares a hand-computed gradient `dt` against PyTorch's `.grad` for the same tensor — **exact** (bit-for-bit), **approximate** (`torch.allclose`), and **maxdiff** (largest absolute disagreement).

In [5]:
# utility function to compare manually- to pytorch-computed gradients
def cmp(s, dt, t):
    ex = torch.all(dt == t.grad).item()
    app = torch.allclose(dt, t.grad)
    maxdiff = (dt - t.grad).abs().max().item()
    print(f'{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff}')

### Initialization

Kaiming-flavored initialization (the `5/3` gain compensates for `tanh`), plus the batch-norm scale `bn_gain` (γ) and shift `bn_bias` (β). Every parameter gets `requires_grad = True` so PyTorch computes the reference gradients.

In [6]:
# initializing the model
n_embedding = 10 # size of the embedding dimension
n_hidden = 64 # number of neurons in the hidden layer
fan_in = n_embedding * block_size
g = torch.Generator().manual_seed(1994) # generator for reproducibility

C = torch.randn((vocab_size, n_embedding),            generator=g)
# layer 1
W1 = torch.randn((fan_in, n_hidden),  generator=g) * (5/3)/(fan_in**0.5)
b1 = torch.randn(n_hidden,                            generator=g) * 0.1
# layer 2
W2 = torch.randn((n_hidden, vocab_size),              generator=g) * 0.1
b2 = torch.randn(vocab_size,                          generator=g) * 0.1
# batch norm parameters
bn_gain = torch.randn((1, n_hidden)) * 0.1 + 1.0
bn_bias = torch.randn((1, n_hidden)) * 0.1

parameters = [C, W1, b1, W2, b2, bn_bias, bn_gain]

for p in parameters:
    p.requires_grad = True
    
print(f"number of parameters = {sum(p.nelement() for p in parameters)}")

number of parameters = 4137


In [7]:
# construct a minibatch
batch_size = 32
n = batch_size
ix = torch.randint(0, Xtrn.shape[0], (n,), generator=g)
Xb, Yb = Xtrn[ix], Ytrn[ix]

## The forward pass, fully decomposed

Every operation gets its own named intermediate tensor, so the chain rule has somewhere to land at each step:

$$
\text{emb} \;\to\; z\_pre = \text{emb}\,W_1 + b_1 \;\to\; \underbrace{\hat z = \frac{z\_pre - \mu}{\sqrt{\sigma^2 + \epsilon}}, \quad z = \gamma\,\hat z + \beta}_{\text{batch norm}} \;\to\; h = \tanh(z) \;\to\; \text{logits} = h\,W_2 + b_2 \;\to\; \text{loss}
$$

The batch statistics use Bessel's correction — variance divides by $n-1$, not $n$. `loss.backward()` then retains the gradient on every intermediate, giving the reference gradients for the whole notebook.

In [8]:
# fully-specified forward pass

# embed training inputs [32, 30]
emb_uncat = C[Xb]
emb = emb_uncat.view(emb_uncat.shape[0], -1) # flattens Xb context window index dimension

# linear layer 1 [32, 64]
z_pre = emb @ W1 + b1 # hidden layer pre-activation

# batch normalization layer 1 [32, 64]
bn_mean = 1/n*z_pre.sum(0, keepdim=True) # sum of each row divided by n: [64, 1]
bn_del = z_pre - bn_mean # delta between pre-activations and row mean: [32, 64]
bn_del2 = bn_del**2
bn_var = 1/(n-1)*(bn_del2).sum(0, keepdim=True) # var = sig^2 = del^2/n, with Bessel's correction /(n-1) instead of n
bn_var_inv = (bn_var + 1e-5)**-0.5 # small scalar to avoid /0
bn_raw = bn_del * bn_var_inv # intermediate term for cleanliness
z = bn_gain * bn_raw + bn_bias # pre-activation after batch normalization

# non-linear layer [32, 64]
h = torch.tanh(z) # hidden layer activation

# linear layer 2 [32, 27]
logits = h @ W2 + b2 # output layer

# cross entropy loss (equivalent to F.cross_entropy(logits, Yb)
logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes # ensures numerical stability
counts = norm_logits.exp() # [32, 27]
counts_sum = counts.sum(1, keepdim=True) # [32, 1]
counts_sum_inv = counts_sum**-1 #
probs = counts * counts_sum_inv # [32, 27]
logprobs = probs.log() # [32, 27]
loss = -logprobs[range(n), Yb].mean() 

# pytorch backward pass
for p in parameters:
    p.grad = None
for t in [logprobs, probs, counts_sum_inv, counts_sum, counts, norm_logits, logit_maxes,
          logits, h, z, bn_raw, bn_var_inv, bn_var, bn_del2, 
          bn_del, bn_mean, z_pre, emb, emb_uncat]:
    t.retain_grad()
loss.backward()
loss

tensor(3.3944, grad_fn=<NegBackward0>)

---

## Exercise 1 — backpropagate the whole thing, one node at a time

Walk the graph in reverse; at each node, the gradient arriving from downstream times the local derivative of the current operation. Where an input reaches the loss through **many** routes, the contributions add:

$$
\frac{\partial \mathcal L}{\partial x} \;=\; \sum_{\text{routes } r} \frac{\partial \mathcal L}{\partial (\text{output}_r)} \cdot \frac{\partial (\text{output}_r)}{\partial x}.
$$

Writing $\partial x \equiv \partial \mathcal L / \partial x$ throughout for the gradient of the loss with respect to $x$.

### Deriving the cross-entropy chain

Every node is the same two questions: *which downstream entries does this one touch, and what is the local derivative along each?* Where a value fans out, sum over the fan-out. Write $\partial x \equiv \partial\mathcal L/\partial x$, and $y_i$ for the correct label of row $i$.

Giving each forward tensor a symbol keeps the algebra readable:

| symbol | code | definition |
|---|---|---|
| $l_{ik}$ | `logits` | — |
| $m_i$ | `logit_maxes` | $\max_k l_{ik}$ |
| $u_{ik}$ | `norm_logits` | $l_{ik} - m_i$ |
| $c_{ik}$ | `counts` | $e^{u_{ik}}$ |
| $s_i$ | `counts_sum` | $\sum_k c_{ik}$ |
| $r_i$ | `counts_sum_inv` | $s_i^{-1}$ |
| $P_{ik}$ | `probs` | $c_{ik}\,r_i$ |
| $\lambda_{ik}$ | `logprobs` | $\ln P_{ik}$ |

**`logprobs`.** The loss picks one entry per row and averages:

$$
\mathcal L = -\frac1n\sum_{i} \lambda_{i,y_i}
\qquad\Longrightarrow\qquad
\partial \lambda_{ik} =
\begin{cases}
-\tfrac1n & k = y_i\\[2pt]
0 & \text{otherwise}
\end{cases}
$$

Entries that never enter the loss get exactly zero gradient.

**`probs`.** Elementwise $\lambda_{ik} = \ln P_{ik}$, so the local derivative is $1/P_{ik}$:

$$
\partial P_{ik} = \partial \lambda_{ik}\cdot \frac{1}{P_{ik}}
$$

**`counts_sum_inv`.** $P_{ik} = c_{ik}\,r_i$, where $r$ has shape `[n,1]` and **broadcasts across all 27 columns**. So $r_i$ reaches the loss through every column of its row — fan-out — and the gradient sums over $k$:

$$
\frac{\partial P_{ik}}{\partial r_i} = c_{ik}
\qquad\Longrightarrow\qquad
\partial r_i = \sum_{k} \partial P_{ik}\;c_{ik}
$$

**`counts_sum`.** From $r_i = s_i^{-1}$, with $\frac{d}{dx}x^{-1} = -x^{-2}$:

$$
\partial s_i = \partial r_i \cdot \big(-s_i^{-2}\big)
$$

**`counts`.** The first node with **two routes**. $c_{ik}$ reaches the loss once directly through $P_{ik}$, and once through $s_i = \sum_k c_{ik}$ (local derivative $1$). The chain rule adds them:

$$
\partial c_{ik}
= \underbrace{\partial P_{ik}\cdot r_i}_{\text{via } P}
\;+\; \underbrace{\partial s_i \cdot 1}_{\text{via } s}
$$

**`norm_logits`.** $c_{ik} = e^{u_{ik}}$, and $\frac{d}{dx}e^x = e^x$ — the local derivative is the forward value itself:

$$
\partial u_{ik} = \partial c_{ik}\cdot c_{ik}
$$

**`logit_maxes`.** $u_{ik} = l_{ik} - m_i$, and $m_i$ broadcasts across the row, so it fans out over all $k$ with local derivative $-1$:

$$
\partial m_i = -\sum_k \partial u_{ik}
$$

**`logits`.** Two routes again: directly into $u_{ik}$ (local derivative $1$), and through the max. The max is a *selector* — it passes the gradient only to the entry it picked:

$$
\frac{\partial m_i}{\partial l_{ik}} = \mathbb{1}\big[k = \arg\max_k l_{ik}\big]
\qquad\Longrightarrow\qquad
\partial l_{ik} = \partial u_{ik} \;+\; \partial m_i\cdot \mathbb{1}\big[k = \arg\max_k l_{ik}\big]
$$

Exercise 2 shows $\partial m_i = 0$, so the second term contributes nothing — but the hand-code carries it anyway, and `cmp` confirms it comes out zero.

In [9]:
# exercise 1: manually compute the derivative of every variable with respect to loss, check against pytorch backward()

dlogprobs = torch.zeros_like(logprobs)
dlogprobs[range(n), Yb] = -1.0/n
cmp('logprobs', dlogprobs, logprobs)

dprobs = dlogprobs/probs
cmp('dprobs', dprobs, probs)

dcounts_sum_inv = (dprobs*counts).sum(1, keepdim=True)
cmp('dcounts_sum_inv', dcounts_sum_inv, counts_sum_inv)

dcounts_sum = -dcounts_sum_inv*counts_sum**-2
cmp('dcounts_sum', dcounts_sum, counts_sum)

dcounts = dprobs*counts_sum_inv
dcounts += torch.ones_like(counts)*dcounts_sum
cmp('dcounts', dcounts, counts)

dnorm_logits = dcounts * counts
cmp('dnorm_logits', dnorm_logits, norm_logits)

dlogit_maxes = -dnorm_logits.sum(1,keepdim=True)
cmp('dlogit_maxes', dlogit_maxes, logit_maxes)

dlogits = torch.zeros_like(logits)
dlogits[range(logits.shape[0]), logits.max(1).indices] = 1.0
dlogits *= dlogit_maxes
dlogits += dnorm_logits
cmp('dlogits', dlogits, logits)

logprobs        | exact: True  | approximate: True  | maxdiff: 0.0
dprobs          | exact: True  | approximate: True  | maxdiff: 0.0
dcounts_sum_inv | exact: True  | approximate: True  | maxdiff: 0.0
dcounts_sum     | exact: True  | approximate: True  | maxdiff: 0.0
dcounts         | exact: True  | approximate: True  | maxdiff: 0.0
dnorm_logits    | exact: True  | approximate: True  | maxdiff: 0.0
dlogit_maxes    | exact: True  | approximate: True  | maxdiff: 0.0
dlogits         | exact: True  | approximate: True  | maxdiff: 0.0


### Backprop through a linear layer: $\text{logits} = h\,W_2 + b_2$

On a small example: $h \in \mathbb{R}^{2\times 5}$, $W \in \mathbb{R}^{5\times 3}$, $b \in \mathbb{R}^{1\times 3}$, giving $l = hW + b \in \mathbb{R}^{2\times 3}$.

Nudging $h_{ij}$ moves only **row $i$** of the logits (rows don't mix in a matmul), carried by **row $j$ of $W$**. It reaches the loss through all three entries of that row, so the chain rule sums over them.

#### Gradient with respect to $h$

$$
\begin{aligned}
\partial h_{11} &= \partial l_{11}W_{11} + \partial l_{12}W_{12} + \partial l_{13}W_{13}\\
\partial h_{12} &= \partial l_{11}W_{21} + \partial l_{12}W_{22} + \partial l_{13}W_{23}\\
\partial h_{13} &= \partial l_{11}W_{31} + \partial l_{12}W_{32} + \partial l_{13}W_{33}\\
\partial h_{14} &= \partial l_{11}W_{41} + \partial l_{12}W_{42} + \partial l_{13}W_{43}\\
\partial h_{15} &= \partial l_{11}W_{51} + \partial l_{12}W_{52} + \partial l_{13}W_{53}\\
\partial h_{21} &= \partial l_{21}W_{11} + \partial l_{22}W_{12} + \partial l_{23}W_{13}\\
\partial h_{22} &= \partial l_{21}W_{21} + \partial l_{22}W_{22} + \partial l_{23}W_{23}\\
\partial h_{23} &= \partial l_{21}W_{31} + \partial l_{22}W_{32} + \partial l_{23}W_{33}\\
\partial h_{24} &= \partial l_{21}W_{41} + \partial l_{22}W_{42} + \partial l_{23}W_{43}\\
\partial h_{25} &= \partial l_{21}W_{51} + \partial l_{22}W_{52} + \partial l_{23}W_{53}
\end{aligned}
$$

In general $\partial h_{ij} = \sum_{b} \partial l_{ib}\, W_{jb}$. The summed index $b$ sits in the **second** slot of both factors; matmul contracts the second slot of the left factor against the first of the right, so $W$ must be transposed:

$$
\boxed{\;\partial h = \partial l \, W^{\top}\;}\qquad [2\times 3]\,[3\times 5] \to [2\times 5].
$$

#### Gradient with respect to $W$

$W_{jb}$ reaches the loss through **column $b$** of the logits, once per batch row, carried by **column $j$ of $h$**:

$$
\begin{aligned}
\partial W_{11} &= \partial l_{11}h_{11} + \partial l_{21}h_{21}, &\partial W_{12} &= \partial l_{12}h_{11} + \partial l_{22}h_{21}, &\partial W_{13} &= \partial l_{13}h_{11} + \partial l_{23}h_{21}\\
\partial W_{21} &= \partial l_{11}h_{12} + \partial l_{21}h_{22}, &\partial W_{22} &= \partial l_{12}h_{12} + \partial l_{22}h_{22}, &\partial W_{23} &= \partial l_{13}h_{12} + \partial l_{23}h_{22}\\
&\;\;\vdots & &\;\;\vdots & &\;\;\vdots\\
\partial W_{51} &= \partial l_{11}h_{15} + \partial l_{21}h_{25}, &\partial W_{52} &= \partial l_{12}h_{15} + \partial l_{22}h_{25}, &\partial W_{53} &= \partial l_{13}h_{15} + \partial l_{23}h_{25}
\end{aligned}
$$

In general $\partial W_{jb} = \sum_{i} \partial l_{ib}\, h_{ij}$. The summed index $i$ (the batch dimension) sits in the **first** slot of both factors, so the *left* factor $h$ is transposed:

$$
\boxed{\;\partial W = h^{\top}\, \partial l\;}\qquad [5\times 2]\,[2\times 3] \to [5\times 3].
$$

#### Gradient with respect to $b$

The bias is broadcast across every row, so each $b_k$ reaches the loss once per batch row with local derivative $1$:

$$
\boxed{\;\partial b = \sum_i \partial l_{i,:} \;=\; \texttt{dlogits.sum(0)}\;}\qquad [2\times 3] \to [1\times 3].
$$

The transpose is not a convention to memorize — it is forced by which index is summed. And **forward fan-out becomes backward summation**: wherever the forward pass copies a value to many places (matmul reuse, a broadcast bias), the backward pass gathers it back by summing.

$$
l = hW + b \quad\Longrightarrow\quad \partial h = \partial l\, W^{\top}, \qquad \partial W = h^{\top} \partial l, \qquad \partial b = \textstyle\sum_i \partial l_{i,:}.
$$

### Deriving the rest of the graph

Symbols again: $x_{ij}$ is `z_pre` (the batch-norm input), $\mu_j,\sigma_j^2$ its column mean and variance, $\delta_{ij} = x_{ij}-\mu_j$ is `bn_del`, $v_j = (\sigma_j^2+\epsilon)^{-1/2}$ is `bn_var_inv`, $\hat z_{ij} = \delta_{ij}v_j$ is `bn_raw`, and $z = \gamma\hat z + \beta$ feeds $h = \tanh z$.

**`tanh`.** With $h = \tanh(z) = \dfrac{e^{z}-e^{-z}}{e^{z}+e^{-z}}$, the quotient rule gives

$$
\frac{dh}{dz} = \frac{(e^{z}+e^{-z})^2 - (e^{z}-e^{-z})^2}{(e^{z}+e^{-z})^2} = 1 - \tanh^2(z) = 1 - h^2
$$

so $\partial z = \partial h \cdot (1 - h^2)$. The local derivative is expressible in the *forward* value — which is why the saturation problem of part 3 shows up here as a vanishing gradient: $h \to \pm 1$ sends $1-h^2 \to 0$.

**`bn_gain`, `bn_bias`.** From $z_{ij} = \gamma_j\,\hat z_{ij} + \beta_j$, with $\gamma,\beta$ of shape `[1,H]` broadcasting down the batch:

$$
\partial \hat z_{ij} = \partial z_{ij}\,\gamma_j,
\qquad
\partial \gamma_j = \sum_i \partial z_{ij}\,\hat z_{ij},
\qquad
\partial \beta_j = \sum_i \partial z_{ij}
$$

The sums over $i$ are the fan-out: one $\gamma_j$ served every row in the batch.

**The batch-norm block, node by node.** Each column is independent, so fix a column and drop $j$:

$$
\hat z_i = \delta_i \, v, \qquad v = (\sigma^2+\epsilon)^{-1/2}, \qquad \sigma^2 = \frac{1}{n-1}\sum_i \delta_i^2, \qquad \delta_i = x_i - \mu, \qquad \mu = \frac1n \sum_i x_i
$$

- $v$ broadcasts down the column, so it fans out:

$$\partial v = \sum_i \partial \hat z_i\,\delta_i$$

- Power rule on $v = (\sigma^2+\epsilon)^{-1/2}$:

$$\partial \sigma^2 = \partial v \cdot \big(-\tfrac12\big)(\sigma^2+\epsilon)^{-3/2}$$

- $\sigma^2$ averages the $\delta_i^2$ **with Bessel's correction**, so $\partial\sigma^2/\partial(\delta_i^2) = \tfrac{1}{n-1}$ for every $i$:

$$\partial(\delta_i^2) = \frac{\partial\sigma^2}{n-1}$$

Differentiate the code you actually wrote — the $n-1$ rides through.

- $\delta_i$ has **two routes**, into $\hat z_i$ and into $\delta_i^2$ (local derivative $2\delta_i$):

$$
\partial \delta_i = \underbrace{\partial \hat z_i \cdot v}_{\text{via } \hat z} \;+\; \underbrace{2\,\delta_i\,\partial(\delta_i^2)}_{\text{via } \sigma^2}
$$

- $\mu$ is subtracted from every entry of the column, so it fans out with local derivative $-1$:

$$\partial\mu = -\sum_i \partial \delta_i$$

- Finally $x_i$ has **two routes** — directly into $\delta_i$, and into $\mu$, which averages it (local derivative $1/n$):

$$
\partial x_i = \partial \delta_i \;+\; \frac{\partial \mu}{n}
$$

That last line is the batch-crossing route, and it is exactly what Exercise 3 collapses.

**`.view`.** A reshape moves no data and does no arithmetic, so its backward is the inverse reshape — `emb` of shape `[n, block_size · n_embedding]` unflattens back to `[n, block_size, n_embedding]`. No arithmetic.

**The embedding lookup.** $\text{emb}_{ij,:} = C_{\,X_{ij},\,:}$ — a *copy* of a row of $C$, where $X_{ij}$ is the character index at position $j$ of context $i$. A row is copied once for every $(i,j)$ that indexes it, which is fan-out, so its gradient sums over every position that read it:

$$
\partial C_{c,:} = \sum_{(i,j)\;:\;X_{ij} = c} \partial\,\text{emb}_{ij,:}
$$

In code that is a **scatter-add** — `+=`, not `=`. Every appearance of the letter "a" anywhere in the batch piles its gradient onto row `ctoi['a']` rather than overwriting it.

In [10]:
dh = dlogits @ torch.transpose(W2,0,1)
dW2 = h.T @ dlogits
db2 = dlogits.sum(0,keepdim=True)

dz = dh * (1.0 - h**2)
dbn_raw = dz * bn_gain
dbn_gain = (dz * bn_raw).sum(0,keepdim=True)
dbn_bias = dz.sum(0,keepdim=True)

dbn_var_inv = (dbn_raw * bn_del).sum(0,keepdim = True)
dbn_var = (-0.5*(bn_var + 1e-5)**-1.5) * dbn_var_inv
dbn_del2 = 1.0/(n-1.0) * dbn_var * torch.ones_like(bn_del2)
dbn_del = (dbn_raw * bn_var_inv) + (2*bn_del*dbn_del2)
dbn_mean = -dbn_del.sum(0,keepdim=True)
dz_pre = dbn_del + (dbn_mean/n * torch.ones_like(z_pre))

dW1 = emb.T @ dz_pre
db1 = dz_pre.sum(0)
demb = dz_pre @ W1.T

demb_uncat = demb.view(demb.shape[0], block_size, n_embedding)

dC = torch.zeros_like(C)
for i in range(batch_size):
    for j in range(block_size):
        dC[Xb[i,j]] += demb_uncat[i,j]

cmp('dh', dh, h)
cmp('dW2', dW2, W2)
cmp('db2', db2, b2)
cmp('dz', dz, z)
cmp('dbn_raw', dbn_raw, bn_raw)
cmp('dbn_gain', dbn_gain, bn_gain)
cmp('dbn_bias', dbn_bias, bn_bias)
cmp('dbn_var_inv', dbn_var_inv, bn_var_inv)
cmp('dbn_var', dbn_var, bn_var)
cmp('dbn_del2', dbn_del2, bn_del2)
cmp('dbn_del', dbn_del, bn_del)
cmp('dbn_mean', dbn_mean, bn_mean)
cmp('dz_pre', dz_pre, z_pre)
cmp('dW1', dW1, W1)
cmp('db1', db1, b1)
cmp('demb', demb, emb)
cmp('demb_uncat', demb_uncat, emb_uncat)
cmp('dC', dC, C)


dh              | exact: True  | approximate: True  | maxdiff: 0.0
dW2             | exact: True  | approximate: True  | maxdiff: 0.0
db2             | exact: True  | approximate: True  | maxdiff: 0.0
dz              | exact: True  | approximate: True  | maxdiff: 0.0
dbn_raw         | exact: True  | approximate: True  | maxdiff: 0.0
dbn_gain        | exact: True  | approximate: True  | maxdiff: 0.0
dbn_bias        | exact: True  | approximate: True  | maxdiff: 0.0
dbn_var_inv     | exact: True  | approximate: True  | maxdiff: 0.0
dbn_var         | exact: True  | approximate: True  | maxdiff: 0.0
dbn_del2        | exact: True  | approximate: True  | maxdiff: 0.0
dbn_del         | exact: True  | approximate: True  | maxdiff: 0.0
dbn_mean        | exact: True  | approximate: True  | maxdiff: 0.0
dz_pre          | exact: True  | approximate: True  | maxdiff: 0.0
dW1             | exact: True  | approximate: True  | maxdiff: 0.0
db1             | exact: True  | approximate: True  | maxdiff:

---

## Exercise 2 — cross-entropy loss and gradient in one line

**Start:** the six-node chain `logits → logit_maxes → norm_logits → counts → counts_sum → counts_sum_inv → probs → logprobs → loss`.
**Goal:** one expression for $\partial\mathcal L/\partial\,\text{logits}$ that mentions none of them.

### Step 1 — collapse the forward pass

Take one example $i$ with correct label $y$, and substitute each forward line into the next. Write $l_{ik}$ for $\text{logits}_{ik}$ and $c_{ik}=e^{l_{ik}}$:

$$
\text{loss}_i
= -\ln P_{iy}
= -\ln\!\left(\frac{c_{iy}}{\sum_k c_{ik}}\right)
= \ln\!\Big(\sum_k c_{ik}\Big) - \ln c_{iy}
$$

using $\ln(a/b) = \ln a - \ln b$. In the second term the outer $\ln$ meets the inner $\exp$ and they cancel, $\ln e^{l_{iy}} = l_{iy}$:

$$
\boxed{\;\text{loss}_i = \ln\Big(\sum_k e^{l_{ik}}\Big) - l_{iy}\;}
$$

The first term is the **log-sum-exp**. There is no identity for $\ln$ of a sum, so the forward simplification stops here — correctly.

### Step 2 — the max-subtraction contributes nothing

Substitute $l_{ik}\to l_{ik}-m_i$ and use $e^{a-b}=e^a e^{-b}$. The constant $e^{-m_i}$ has no $k$ in it, so it factors straight out of the sum:

$$
\ln\Big(\sum_k e^{\,l_{ik}-m_i}\Big) - (l_{iy}-m_i)
= \ln\Big(e^{-m_i}\sum_k e^{l_{ik}}\Big) - l_{iy} + m_i
= \underbrace{-m_i}_{\text{from } \ln e^{-m_i}} + \ln\Big(\sum_k e^{l_{ik}}\Big) - l_{iy} + m_i
$$

The two $m_i$ cancel and we are back to the boxed loss. Subtracting the max changes neither the loss nor any gradient — it is pure numerical hygiene, which is *why* it can be dropped when differentiating.

### Step 3 — differentiate

Differentiate the boxed loss with respect to one logit $l_{ij}$, term by term.

*The label term* $-l_{iy}$ depends on $l_{ij}$ only when $j=y$, contributing $-\mathbb{1}[j=y_i]$.

*The log-sum term*, by the chain rule on $\ln(\cdot)$:

$$
\frac{\partial}{\partial l_{ij}}\ln\Big(\sum_k e^{l_{ik}}\Big)
= \frac{1}{\sum_k e^{l_{ik}}}\cdot\frac{\partial}{\partial l_{ij}}\sum_k e^{l_{ik}}
$$

Inside that sum, only the $k=j$ term depends on $l_{ij}$ — every other term is a constant and dies. The sum collapses to a single survivor $e^{l_{ij}}$:

$$
= \frac{e^{l_{ij}}}{\sum_k e^{l_{ik}}} = \text{probs}_{ij}
$$

which is the softmax probability **already computed in the forward pass**. Adding the two terms:

$$
\boxed{\;\frac{\partial\,\text{loss}_i}{\partial l_{ij}} = \text{probs}_{ij} - \mathbb{1}[j = y_i]\;}
$$

and since $\mathcal L = \frac1n\sum_i \text{loss}_i$, the batched gradient is $\tfrac1n(\text{probs} - \text{onehot})$.

**Predicted minus target.** Drain probability from the wrong classes, pile it on the right one, in proportion to how wrong the model currently is.

In [11]:
# exercise 2: calculating loss from logits fast
# slow method:
# 
# # linear layer 2 [32, 27]
# logits = h @ W2 + b2 # output layer
# 
# # cross entropy loss (equivalent to F.cross_entropy(logits, Yb)
# logit_maxes = logits.max(1, keepdim=True).values
# norm_logits = logits - logit_maxes # ensures numerical stability
# counts = norm_logits.exp() # [32, 27]
# counts_sum = counts.sum(1, keepdim=True) # [32, 1]
# counts_sum_inv = counts_sum**-1 #
# probs = counts * counts_sum_inv # [32, 27]
# logprobs = probs.log() # [32, 27]
# loss = -logprobs[range(n), Yb].mean() 

# calculated by algebra:
# Loss = 1/n * -∑(ln(P[y]))
# Loss = 1/n * ∑ ln(∑_k e^l_ik) - l_iy
# and derivative d/dl_i = Pi if i =/= y and Pi - 1.0 if i == y


dlogits = 1.0/n * (probs - F.one_hot(Yb, vocab_size)).float()
cmp('dlogits', dlogits, logits)

dlogits         | exact: False | approximate: True  | maxdiff: 5.820766091346741e-09


---

## Exercise 3 — batch normalization backward in one line

**Start:** the seven-node chain `z_pre → bn_mean → bn_del → bn_del2 → bn_var → bn_var_inv → bn_raw`.
**Goal:** one expression taking the gradient at the block's output straight to $\partial x$.

Symbols: $x_i$ is `z_pre`, $\hat z_i$ is `bn_raw`, $\gamma$ is `bn_gain`, $v = (\sigma^2+\epsilon)^{-1/2}$ is `bn_var_inv`.

### Why this one is hard: the Jacobian is full

`tanh` is elementwise — output $a$ depends only on input $a$, so nudging one input moves exactly one output. Batch norm breaks that. The mean and variance are computed **down the column** (`.sum(0)`), so they depend on every entry in the column, and every output subtracts that shared mean and divides by that shared standard deviation. Nudging a single $x_i$ moves **every** output in its column, through three channels:

1. **directly**, via its own numerator $x_i - \mu$;
2. **through the mean** $\mu$, which shifts every output;
3. **through the variance** $\sigma^2$, which rescales every output.

Lay the sensitivities out as a grid $J_{ai} = \partial \hat z_a/\partial\,x_i$. An elementwise op makes this grid **diagonal**. Batch norm's is **full** — every cell nonzero. That fullness is the whole difficulty, and it is why the answer will be a sum over the entire column.

One simplification first: the statistics are per-column, so the 64 columns are **completely independent** — 64 parallel batch-norm ops that happen to sit side by side. Fix one column, drop the column index, and solve a clean $n$-in / $n$-out problem; the result applies verbatim to all 64.

### The two atoms

With $\mu = \frac1n\sum_a x_a$ and, with Bessel's correction, $\sigma^2 = \frac{1}{n-1}\sum_a (x_a - \mu)^2$:

**Mean.** Only the $a=i$ term of the sum depends on $x_i$, so

$$
\frac{\partial \mu}{\partial\,x_i} = \frac1n
$$

**Variance.** Each term depends on $x_i$ twice over — directly when $a=i$, and through the shared $\mu$. Chain-ruling the square and then substituting the mean atom:

$$
\frac{\partial \sigma^2}{\partial\,x_i}
= \frac{1}{n-1}\sum_a 2\,(x_a - \mu)\cdot\frac{\partial (x_a - \mu)}{\partial\,x_i}
= \frac{2}{n-1}\sum_a (x_a - \mu)\Big(\mathbb{1}[a{=}i] - \tfrac1n\Big)
$$

Distribute the sum. The $\mathbb{1}[a{=}i]$ picks out the single term $a=i$; the $-\tfrac1n$ term multiplies the sum of all deviations:

$$
= \frac{2}{n-1}\Big[(x_i - \mu) - \tfrac1n\underbrace{\textstyle\sum_a (x_a - \mu)}_{=\,0}\Big]
= \boxed{\;\frac{2}{n-1}(x_i - \mu)\;}
$$

The second term vanishes because **deviations from the mean sum to zero** — the mean is by definition the balance point. This identity, $\sum_a(x_a - \mu) = 0$, does the real work here and again below.

Note the $\frac{2}{n-1}$: differentiate the variance you actually *wrote*. The textbook $\frac1n$ variance would give $\frac{2}{n}$, and every gradient downstream would be quietly wrong.

### One cell of the Jacobian

Differentiate a general output $\hat z_a = (x_a - \mu)\,(\sigma^2+\epsilon)^{-1/2}$ with respect to a general input $x_i$. Keeping the two indices **distinct** is what captures the off-diagonal coupling — set $a=i$ too early and the whole difficulty disappears.

It is a product of two factors, each depending on $x_i$, so the product rule gives two terms:

$$
\frac{\partial \hat z_a}{\partial\,x_i}
= \underbrace{\frac{\partial (x_a - \mu)}{\partial\,x_i}\,(\sigma^2+\epsilon)^{-1/2}}_{\text{numerator moves}}
\;+\; \underbrace{(x_a - \mu)\,\frac{\partial (\sigma^2+\epsilon)^{-1/2}}{\partial\,x_i}}_{\text{denominator moves}}
$$

The first factor is the mean atom: $\;\partial(x_a - \mu)/\partial\,x_i = \mathbb{1}[a{=}i] - \tfrac1n$.

The second needs the power rule and then the variance atom:

$$
\frac{\partial (\sigma^2+\epsilon)^{-1/2}}{\partial\,x_i}
= -\tfrac12(\sigma^2+\epsilon)^{-3/2}\cdot\frac{\partial \sigma^2}{\partial\,x_i}
= -\tfrac12(\sigma^2+\epsilon)^{-3/2}\cdot\frac{2}{n-1}(x_i - \mu)
= -\frac{(x_i - \mu)}{n-1}(\sigma^2+\epsilon)^{-3/2}
$$

Substituting both back:

$$
\frac{\partial \hat z_a}{\partial\,x_i}
= \Big(\mathbb{1}[a{=}i] - \tfrac1n\Big)(\sigma^2+\epsilon)^{-1/2}
\;-\; \frac{(x_a - \mu)(x_i - \mu)}{n-1}\,(\sigma^2+\epsilon)^{-3/2}
$$

Now pull one factor of $(\sigma^2+\epsilon)^{-1/2}$ out of both terms. That leaves $(\sigma^2+\epsilon)^{-1}$ behind in the second, which pairs with the two deviations — and $(x_a-\mu)(\sigma^2+\epsilon)^{-1/2}$ is just $\hat z_a$, likewise $\hat z_i$:

$$
\boxed{\;\frac{\partial \hat z_a}{\partial\,x_i} = (\sigma^2+\epsilon)^{-1/2}\Big[\,\mathbb{1}[a{=}i] \;-\; \tfrac1n \;-\; \tfrac{1}{n-1}\,\hat z_a\,\hat z_i\,\Big]\;}
$$

The three bracket terms are exactly the three routes: the Kronecker delta is the **direct** route (diagonal only), $-\tfrac1n$ is the **mean** route (every $a$), and $-\tfrac{1}{n-1}\hat z_a \hat z_i$ is the **variance** route (every $a$). The last two are what fill the off-diagonal.

### Assembling the full gradient

The grid being full, $x_i$ reaches the loss through *every* output in its column, so the chain rule sums over all of them:

$$
\frac{\partial \mathcal L}{\partial\,x_i} = \sum_a \frac{\partial \mathcal L}{\partial \hat z_a}\,\frac{\partial \hat z_a}{\partial\,x_i}
$$

Let $d\hat z_a = \partial\mathcal L/\partial \hat z_a$ be the gradient arriving at the normalized values — which is just the block-output gradient times the gain, $d\hat z = \gamma\,\partial z$. Substitute the boxed cell, pull the $a$-independent $(\sigma^2+\epsilon)^{-1/2}$ out front, and distribute the sum across the three bracket terms:

$$
\frac{\partial \mathcal L}{\partial\,x_i}
= (\sigma^2+\epsilon)^{-1/2}\Big[\underbrace{\sum_a d\hat z_a\,\mathbb{1}[a{=}i]}_{\text{one survivor}} \;-\; \underbrace{\tfrac1n\sum_a d\hat z_a}_{\text{sum of incoming}} \;-\; \underbrace{\tfrac{\hat z_i}{n-1}\sum_a d\hat z_a\,\hat z_a}_{\text{incoming, weighted by } \hat z}\Big]
$$

The first sum collapses to the single term $d\hat z_i$. Folding the gain back out front ($d\hat z = \gamma\,\partial z$):

$$
\boxed{\;
\frac{\partial \mathcal L}{\partial\,x_i}
= \gamma\,(\sigma^2+\epsilon)^{-1/2}\Big[\,\partial z_i \;-\; \tfrac1n\textstyle\sum_a \partial z_a \;-\; \tfrac{\hat z_i}{n-1}\textstyle\sum_a \partial z_a\,\hat z_a\,\Big]
\;}
$$

The two sums are **single numbers per column** — computed once, reused for every row — while $\partial z_i$ and $\hat z_i$ carry the free row index and stay full arrays. In code that is two `.sum(0)` reductions plus elementwise arithmetic: `bn_gain` is $\gamma$, `bn_var_inv` is $(\sigma^2+\epsilon)^{-1/2}$, `dz` is $\partial z$, and `bn_raw` is $\hat z$.

In [12]:
# exercise 3: batch normalization shortcut, goal - compute dL/dzp with dL/dz, bypass the whole batch norm layer
# old method:

# z_pre = emb @ W1 + b1 # hidden layer pre-activation
#### skip from here...
# bn_mean = 1/n*z_pre.sum(0, keepdim=True) # sum of each row divided by n: [64, 1]
# bn_del = z_pre - bn_mean # delta between pre-activations and row mean: [32, 64]
# bn_del2 = bn_del**2
# bn_var = 1/(n-1)*(bn_del2).sum(0, keepdim=True) # var = sig^2 = del^2/n, with Bessel's correction /(n-1) instead of n
# bn_var_inv = (bn_var + 1e-5)**-0.5 # small scalar to avoid /0
# bn_raw = bn_del * bn_var_inv # intermediate term for cleanliness
#### ... to here!
# z = bn_gain * bn_raw + bn_bias # pre-activation after batch normalization

dz_pre = bn_gain * bn_var_inv * (dz - (1/n)*dz.sum(0,keepdim=True) - (bn_raw/(n-1))*(dz*bn_raw).sum(0,keepdim=True))
cmp('dz_pre', dz_pre, z_pre)

dz_pre          | exact: False | approximate: True  | maxdiff: 9.313225746154785e-10
